## Code 1: Inventory Planning Problem Lecture 15 
Company X must decide how many smartphones to produce in each of the next four quarters.
- The demand during each quarter is (Q1: 35k, Q2: 50k, Q3: 30k, Q4: 60k).
- Company X must meet demands on time (by the end of each quarter).
- At the beginning of Q1, Company X has 10k smart-phones in inventory.
- It can produce up to 40k smartphones in a quarter with regular production at the cost of $200/phone, or overtime production at the cost of $250/phone.
- At the end of each quarter, there is a $50/phone inventory holding cost.

Questions:
1. How should we define the decision variables?
2. What is the objective?
3. What are the constraints?

In [1]:
# pip install gurobipy==11.0.0

In [11]:
# -*- coding: utf-8 -*-
"""
Created on Wed March 20 13:53:10 2024

@author: npatankar
"""
# Step 1: Add the Basic Packages
import gurobipy as gp
#from gurobipy import Model, GRB


# Step 2: Setup the parameters for the model
# quarters = ['Q1', 'Q2', 'Q3', 'Q4']
# demand = {'Q1': 35000, 'Q2': 50000, 'Q3': 30000, 'Q4': 60000} # Demand for each Quarter

quarters = [1, 2, 3, 4]
demand = {1: 35000, 2: 50000, 3: 30000, 4: 60000} # Demand for each Quarter

initial_inventory = 10000   # Starting or initial Inventory
regular_capacity = 40000    # Upper Bound in Regular Production
regular_cost = 200          # Cost of Regular Production
overtime_cost = 250         # Cost of Overtime Production
holding_cost = 50           # Cost of Overtime Production

In [12]:
holding_cost

50

Definition of Variables
- Let $X_i~~\forall i \in \{1,2,3,4\}$ be the number of smartphones produced in regular production in quarters $Q_q~~\forall q \in \{1,2,3,4\}$.
- Let $Y_i~~\forall i \in \{1,2,3,4\}$ be the number of smartphones produced in overtime production in quarters $Q_q~~\forall q \in \{1,2,3,4\}$.
- Let $I_i~~\forall i \in \{1,2,3,4\}$ be the number of smartphones held in inventory at the end of quarters $Q_q~~\forall q \in \{1,2,3,4\}$.

In [14]:
# Step 3: Setup a LP model 
model = gp.Model("InventoryPlanning")

# Step 4: Add Decision Variables to the Model
# Gurubi Syntax: addVar(lb=0.0, ub=float('inf'), obj=0.0, vtype=GRB.CONTINUOUS, name='', column=None)
# Gurobi Syntax:  addVars(*indices, lb=0.0, ub=float('inf'), obj=0.0, vtype=GRB.CONTINUOUS, name='')
# vtype – (optional) Variable type for new variable (GRB.CONTINUOUS, GRB.BINARY, GRB.INTEGER
# Gurobi URL: https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.addVars

# Regular production in each quarter
# X1 = model.addVar(name="X", lb=0, ub=regular_capacity)
X = model.addVars(quarters, name="X", lb=0, ub=regular_capacity)

# Overtime production in each quarter
Y = model.addVars(quarters, name="Y", lb=0)

# Inventory at the end of each quarter
I = model.addVars(quarters, name="I", lb=0)

model.update()
print(model)

Set parameter TokenServer to value "golden.bu.binghamton.edu"
<gurobi.Model Continuous instance InventoryPlanning: 0 constrs, 12 vars, Parameter changes: TokenServer=golden.bu.binghamton.edu>


In [15]:
for var in model.getVars():
    print(var.LB, "<=", var.VarName,"<=", var.UB)

0.0 <= X[1] <= 40000.0
0.0 <= X[2] <= 40000.0
0.0 <= X[3] <= 40000.0
0.0 <= X[4] <= 40000.0
0.0 <= Y[1] <= inf
0.0 <= Y[2] <= inf
0.0 <= Y[3] <= inf
0.0 <= Y[4] <= inf
0.0 <= I[1] <= inf
0.0 <= I[2] <= inf
0.0 <= I[3] <= inf
0.0 <= I[4] <= inf


Objective Function

Minimize the total cost, which includes regular production costs, overtime production costs, and inventory holding costs, while meeting customers demand

$$\min 200 X_i + 250 Y_i + 50 I_i$$

In [17]:
# Objective function: Minimize total cost
# setObjective(expr, sense=None)
# sense – (optional) Optimization sense (GRB.MINIMIZE for minimization, GRB.MAXIMIZE for maximization)
# https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.setObjective

# total_cost = (
#     sum(X[q] * regular_cost for q in quarters) +
#     sum(Y[q] * overtime_cost for q in quarters) +
#     sum(I[q] * holding_cost for q in quarters)
# )
total_cost = (
    regular_cost*(X[1] + X[2] + X[3]+X[4]) +
    overtime_cost*(Y[1] + Y[2] + Y[3]+Y[4])  +
    holding_cost*(I[1] + I[2] + I[3]+I[4]) 
)
model.setObjective(total_cost, gp.GRB.MINIMIZE)

model.update()
print(model)

<gurobi.Model Continuous instance InventoryPlanning: 0 constrs, 12 vars, Parameter changes: TokenServer=golden.bu.binghamton.edu>


In [18]:
print("Objective Function:")
print(model.getObjective())


Objective Function:
200.0 X[1] + 200.0 X[2] + 200.0 X[3] + 200.0 X[4] + 250.0 Y[1] + 250.0 Y[2] + 250.0 Y[3] + 250.0 Y[4] + 50.0 I[1] + 50.0 I[2] + 50.0 I[3] + 50.0 I[4]


Constraints

Let $I0$ be the number of smartphones in inventory at the start of quarter 1. This is a given constant and is equal to
10,000. Constraints are:
$$\begin{align}
I1 &= I0 + x1 + y1 − 35,000 \\
I2 &= I1 + x2 + y2 − 50,000 \\
I3 &= I2 + x3 + y3 − 30,000 \\
I4 &= I3 + x4 + y4 − 60,000 \\
x1 &≤ 40,000 \\
x2 &≤ 40,000 \\
x3 &≤ 40,000 \\
x4 &≤ 40,000 \\
&x1, x2, x3, x4, y1, y2, y3, y4 , I1, I2, I3, I4 ≥ 0 \\
\end{align}$$

OR simply:
$$\begin{align}
I_i &= I_{i-1} + X_i + Y_i − \text{Demand}_i, &\quad \forall i \in \{1,2,3,4\} \\
X_i &≤ 40,000, &\quad \forall i \in \{1,2,3,4\}\\
&X_i, Y_i , I_i ≥ 0, &\quad \forall i \in \{1,2,3,4\} \\
\end{align}$$

In [20]:
# Step 6: Add Constraints to the Model
# Gurobi Syntax: addConstrs(generator, name='')
# Gurobi Syntax: addConstr(constr, name='')
# Gurobi Url: https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.addConstrs

# Initial inventory constraint for Q1
# model.addConstrs(
#     (initial_inventory + X[i] + Y[i] - I[i] == demand[i] if i == 1 
#      else I[i-1] + X[i] + Y[i] - I[i] == demand[i] for i in range(1, 5)),
#     name="Demand"
# )

model.addConstr(initial_inventory + X[1] + Y[1] - I[1] == demand[1], name="Demand1")
model.addConstr(I[1] + X[2] + Y[2] - I[2] == demand[2], name="Demand2")
model.addConstr(I[2] + X[3] + Y[3] - I[3] == demand[3], name="Demand3")
model.addConstr(I[3] + X[4] + Y[4] - I[4] == demand[4], name="Demand4")

model.update()
print(model)


<gurobi.Model Continuous instance InventoryPlanning: 4 constrs, 12 vars, Parameter changes: TokenServer=golden.bu.binghamton.edu>


In [21]:
print("Constraints in the model:")
for constr in model.getConstrs():
    print(constr.ConstrName,": ",model.getRow(constr), constr.Sense, constr.RHS)

Constraints in the model:
Demand1 :  X[1] + Y[1] + -1.0 I[1] = 25000.0
Demand2 :  X[2] + Y[2] + I[1] + -1.0 I[2] = 50000.0
Demand3 :  X[3] + Y[3] + I[2] + -1.0 I[3] = 30000.0
Demand4 :  X[4] + Y[4] + I[3] + -1.0 I[4] = 60000.0


In [22]:
# Optimize the model - Min is default
model.optimize()

# Print the results
if model.status == gp.GRB.OPTIMAL:
    print("Optimal solution found:")
    for q in quarters:
        print(f"{q}:")
        print(f"  Regular Production: {X[q].X}")
        print(f"  Overtime Production: {Y[q].X}")
        print(f"  Inventory: {I[q].X}")
    print(f"Total Cost: ${model.ObjVal:,.2f}")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (22631.2))

CPU model: Intel(R) Xeon(R) Silver 4208 CPU @ 2.10GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 4 rows, 12 columns and 15 nonzeros
Model fingerprint: 0xdad755db
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+01, 3e+02]
  Bounds range     [4e+04, 4e+04]
  RHS range        [3e+04, 6e+04]
Presolve time: 0.01s
Presolved: 4 rows, 12 columns, 15 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   2.062500e+04   0.000000e+00      0s
       5    3.4500000e+07   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.02 seconds (0.00 work units)
Optimal objective  3.450000000e+07
Optimal solution found:
1:
  Regular Production: 35000.0
  Overtime Production: 0.0
  Inventory: 10000.0
2:
  Regular Production: 40000.0
  Overtim

In [23]:
# Open a file to write the results
with open("inventory_planning_results.txt", "w") as f:
    if model.status == gp.GRB.OPTIMAL:
        # Print the objective function to the file
        f.write("Objective Function:\n")
        f.write(str(model.getObjective()) + "\n\n")

        # Print variables with their ranges and values at optimality to the file
        f.write("Variables:\n")
        for var in model.getVars():
            f.write(f"{var.LB} <= {var.VarName} <= {var.UB},    Value = {var.X}\n")
 
        # Print the optimal solution to the file
        f.write("Optimal solution found:\n")
        f.write(f"  Regular Production: {[X[i].X for i in quarters]}\n")
        f.write(f"  Overtime Production: {[Y[i].X for i in quarters]}\n")
        f.write(f"  Inventory: {[I[i].X for i in quarters]}\n")
        f.write(f"Total Cost: ${model.ObjVal:,.2f}\n")
    else:
        f.write("No optimal solution found.\n")